# 7 (Bonus): Imbalanced Learning

**Author:** Dr Rob Lyon 

**Version:** 1.0

## Code & License
Copyright © 2026 Robert Lyon. All Rights Reserved.

This notebook and all associated materials are the intellectual property of the author.

Permission is granted solely to read, study, and analyse this material for personal educational purposes. No other rights are granted.

Without the prior written consent of the author, you may not:

* Copy, reproduce, redistribute, publish, transmit, or display this work in whole or in part.
* Modify, adapt, transform, translate, or create derivative works based on this material.
* Incorporate any portion of this work into another project, publication, product, model, dataset, or codebase.
* Use this material for commercial purposes.
* Remove or alter this copyright notice.

All intellectual property rights, including copyright and any derivative rights, remain exclusively vested in the author.

Access to this material does not constitute a transfer of ownership, license, or any other intellectual property rights except as expressly stated above.

---

## Table of Contents

1. [Introduction](#1.-Introduction)
2. [Useful Links](#2.-Useful-Links)
3. [Libraries and Environment Setup](#3.-Libraries-and-Environment-Setup)
4. [What Is Class Imbalance?](#4.-What-Is-Class-Imbalance?)
5. [Why Imbalance Is Hard](#5.-Why-Imbalance-Is-Hard)
    - [5.1 Absolute Rarity](#5.1-Absolute-Rarity)
    - [5.2 Small Disjuncts](#5.2-Small-Disjuncts)
    - [5.3 Class Overlap](#5.3-Class-Overlap)
    - [5.4 Seeing All Three Together](#5.4-Seeing-All-Three-Together)
6. [How the Problem Manifests](#6.-How-the-Problem-Manifests)
7. [Imbalanced Learning, Anomaly Detection, and One-Class Learning](#7.-Imbalanced-Learning,-Anomaly-Detection,-and-One-Class-Learning)
8. [Data-Level Approaches](#8.-Data-Level-Approaches)
    - [8.1 Random Undersampling](#8.1-Random-Undersampling)
    - [8.2 Random Oversampling](#8.2-Random-Oversampling)
    - [8.3 Stratification](#8.3-Stratification)
    - [8.4 Artificial Rare-Class Generation: SMOTE and Borderline-SMOTE](#8.4-Artificial-Rare-Class-Generation:-SMOTE-and-Borderline-SMOTE)
9. [Algorithm-Level Approaches](#9.-Algorithm-Level-Approaches)
    - [9.1 Changing the Learning Objective](#9.1-Changing-the-Learning-Objective)
    - [9.2 Cost-Sensitive Learning](#9.2-Cost-Sensitive-Learning)
    - [9.3 A Worked Example: Cost-Sensitive Naïve Bayes](#9.3-A-Worked-Example:-Cost-Sensitive-Naïve-Bayes)
10. [Summary](#10.-Summary)
11. [References](#11.-References)

---

## 1. Introduction

Every classifier we have built so far in this series has been trained on data where the classes were reasonably balanced: roughly equal numbers of apples and oranges, roughly equal numbers of benign and malignant tumours. Most real-world classification problems do not look like this. Fraud is rare. Disease is rare. Equipment failure is rare. Network intrusions are rare. In each of these cases the class we care about most, the one we are building the model to catch, is also the class with the fewest examples in the training data.

This notebook looks at what happens when the classes in a dataset are not balanced, why standard classifiers tend to fail quietly on the minority class, and what can be done about it. We will work through the taxonomy of imbalance proposed by He and Garcia (2009), one of the most widely cited reviews in this area, and use a simple synthetic example to show what absolute rarity, small disjuncts, and class overlap actually look like in a scatter plot. We will then connect imbalanced learning to the closely related problems of anomaly detection and one-class learning, before surveying the two broad families of solution: data-level approaches (resampling) and algorithm-level approaches (changing what the learner optimises for).

### Learning Objectives

By the end of this notebook you should be able to:

- Define class imbalance and explain why accuracy is a misleading metric when classes are skewed.
- Distinguish between absolute rarity, small disjuncts, and class overlap as separate sources of difficulty in imbalanced data.
- Explain how imbalance manifests as poor recall on the minority class, and relate this to the precision-recall tradeoff.
- Describe the relationship between imbalanced learning, anomaly/outlier detection, and one-class learning.
- Apply random undersampling and random oversampling using scikit-learn and `imbalanced-learn`.
- Explain how SMOTE and Borderline-SMOTE generate synthetic minority examples, and discuss their respective strengths and weaknesses.
- Explain the principle behind cost-sensitive learning and implement a simple cost-sensitive version of Naïve Bayes.

---


---

## 2. Useful Links

| Link | Description |
|------|-------------|
| [He & Garcia (2009), Learning from Imbalanced Data](https://doi.org/10.1109/TKDE.2008.239) | The standard reference survey on imbalanced learning: nature of the problem, data-level and algorithm-level solutions, and evaluation metrics. |
| [Chawla et al. (2002), SMOTE: Synthetic Minority Over-sampling Technique](https://doi.org/10.1613/jair.953) | The original SMOTE paper, introducing synthetic oversampling via interpolation between minority neighbours. |
| [imbalanced-learn documentation](https://imbalanced-learn.org/stable/) | The standard scikit-learn-compatible library for resampling: random under/oversampling, SMOTE and its variants, and combined methods. |
| [imbalanced-learn: SMOTE API reference](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html) | Full parameter reference for the `SMOTE` class used in this notebook. |
| [imbalanced-learn: BorderlineSMOTE API reference](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.BorderlineSMOTE.html) | Full parameter reference for the `BorderlineSMOTE` class. |
| [scikit-learn: `class_weight` user guide](https://scikit-learn.org/stable/modules/svm.html#unbalanced-problems) | Background on how scikit-learn estimators use `class_weight='balanced'` to implement cost-sensitive learning. |


---


---

## 3. Libraries and Environment Setup

🔙 [Back to Table of Contents](#Table-of-Contents)

### 3.1 Working Environment

To run the code in this notebook, you will need a Python environment with the required libraries listed below installed.

The easiest way to get started is to use the **Project IO** Docker container, which provides a fully configured and tested environment containing all required Python packages and supporting dependencies. Instructions for downloading and running the container can be found in the **project-io** GitHub repository:

https://github.com/scienceguyrob/project-io

Using the Docker container ensures that the notebooks run in exactly the same environment in which they were developed and tested.

If you prefer not to use Docker, a good alternative is to install the [Anaconda Distribution](https://www.anaconda.com/products/distribution). You can then create a new Python environment and install the required libraries using either `conda install` or `pip install`.

### 3.2 Libraries Used in This Notebook

All sections of this notebook require the libraries listed below.

| Library | Purpose | Documentation |
|---------|---------|---------------|
| **NumPy** (`numpy`) | Numerical arrays, random number generation, and mathematical operations. Used throughout for data creation and computation. | [numpy.org](https://numpy.org/doc/stable/) |
| **Matplotlib** (`matplotlib.pyplot`) | The standard Python plotting library. Used to produce all scatter plots and comparison figures. | [matplotlib.org](https://matplotlib.org/stable/) |
| **scikit-learn** (`sklearn`) | Provides `GaussianNB`, dataset generators, train/test splitting, and evaluation metrics. | [scikit-learn.org](https://scikit-learn.org/stable/) |
| **imbalanced-learn** (`imblearn`) | A scikit-learn-compatible library implementing random under/oversampling, SMOTE, and Borderline-SMOTE. | [imbalanced-learn.org](https://imbalanced-learn.org/stable/) |

### 3.3 A Note on `imbalanced-learn`

`imbalanced-learn` (imported as `imblearn`) is not part of the standard scikit-learn install and needs to be installed separately:

```python
pip install imbalanced-learn
```

It follows the same `fit`/`fit_resample` pattern as scikit-learn transformers, and every sampler used in this notebook (`RandomUnderSampler`, `RandomOverSampler`, `SMOTE`, `BorderlineSMOTE`) is a drop-in object that takes `X` and `y` and returns a resampled `X` and `y`.

### 3.4 Imports

All library imports for this notebook are placed in the single cell below. This is deliberate best practice: keeping all imports in one place makes it easy to see at a glance what is required and avoids confusing `NameError` exceptions that arise when an import cell is accidentally skipped.

> ⚠️ **You must run the cell below before executing any other code cell in this notebook.**

In [ ]:
# Listing 1.
# ─────────────────────────────────────────────────────────────────────────────
# All library imports for this notebook are placed here for clarity.
# Run this cell first before executing any code.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np                          # numerical arrays and random number generation
import pandas as pd                         # to load and tabulate data
import matplotlib.pyplot as plt             # plotting and visualisation
import matplotlib.patches as mpatches       # custom legend handles

import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 50

# scikit-learn: datasets and model selection
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split

# scikit-learn: classifier and evaluation metrics
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score, f1_score,
)

# imbalanced-learn: resampling techniques
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE, BorderlineSMOTE

# Shared random number generator for reproducibility
rng = np.random.default_rng(42)   # seed 42 ensures the same data every run

# Confirm successful import
print('Libraries loaded successfully.')


---

## 4. What Is Class Imbalance?

🔙 [Back to Table of Contents](#Table-of-Contents)

A dataset is **imbalanced** when the classes it contains are not represented in roughly equal proportions. In a binary classification problem this usually means one class, the **majority class**, vastly outnumbers the other, the **minority class**. The degree of imbalance is often summarised by the **imbalance ratio**, the number of majority examples divided by the number of minority examples. A dataset with 9,900 negative cases and 100 positive cases has an imbalance ratio of 99:1.

Imbalance is everywhere in applied machine learning:

- **Fraud detection**: the overwhelming majority of transactions are legitimate; fraudulent ones might be 1 in 1,000 or rarer.
- **Medical diagnosis**: most patients screened for a rare disease do not have it.
- **Manufacturing defect detection**: most items coming off a production line pass quality control.
- **Network intrusion detection**: most network traffic is benign.
- **Churn prediction**: most customers do not cancel their subscription in any given month.

In every one of these cases, the minority class is also the class of greatest practical interest. Missing a fraudulent transaction, a malignant tumour, or a network intrusion is usually far more costly than a false alarm on the majority class. This is precisely why imbalance matters: the class we care about most is also the class a standard classifier is least equipped to learn.

### Why Accuracy Lies

Suppose we build a classifier for a dataset with a 99:1 imbalance ratio: 9,900 negative examples and 100 positive examples. A classifier that simply predicts "negative" for every single input, without looking at the data at all, achieves **99% accuracy**. It is also completely useless: it catches none of the positive cases we built the model to find.

This is the central trap of imbalanced learning. A model can score very highly on the most commonly reported metrics (more in later notebooks) while being entirely blind to the class that matters. This is why, throughout this notebook, we report multiple evaluation metrics such as **precision, recall, and F1-score per class**, rather than relying on overall accuracy, exactly as we did with `classification_report` in earlier notebooks. Recall on the minority class in particular tells us what fraction of the genuinely positive cases the model actually catches, which is usually the number that matters most in practice. Please keep in mind that notebooks 11 and 12 will cover these metrics inmuch more detail.


---

## 5. Why Imbalance Is Hard

🔙 [Back to Table of Contents](#Table-of-Contents)

It is tempting to think of class imbalance as a single, simple problem: "there isn't enough data for the rare class". In practice the difficulty is more nuanced than that. He and Garcia (2009), in one of the most widely cited reviews of this area, identify several intrinsic data characteristics that compound the raw imbalance ratio and make learning substantially harder. Three of the most important are **absolute rarity**, **small disjuncts**, and **class overlap**. A dataset can suffer from any one of these, or, as is common in practice, from all three simultaneously.

### 5.1 Absolute Rarity

A class suffers from **absolute rarity** when there simply are not enough examples of it in the data, regardless of how that class is distributed relative to the majority class. This is distinct from a high imbalance ratio: a dataset could have an imbalance ratio of 10:1 with only 20 minority examples in total, which is a case of severe absolute rarity, or a ratio of 10:1 with 100,000 minority examples, which is a much easier problem despite having the same ratio. With very few minority examples, any learning algorithm struggles to estimate what the minority class actually looks like: a handful of points give a poor picture of the true shape, spread, and boundaries of the class, and the model has very little signal to learn from no matter how cleverly it is built.

### 5.2 Small Disjuncts

Even when a class has a reasonable number of examples overall, those examples may not form a single, compact region of the feature space. Instead, the class concept can be split into several smaller, disconnected sub-clusters, a phenomenon Holte et al. (1989) and later Japkowicz (2004) termed **small disjuncts**. Each individual sub-cluster, on its own, may contain very few points, even if the class as a whole is reasonably sized. A classifier that is only sensitive to the largest, densest part of the minority class can fail badly on these smaller pockets, which behave like miniature absolute-rarity problems nested inside an otherwise adequately sized class.

### 5.3 Class Overlap

**Class overlap** occurs when examples from different classes sit close together, or directly on top of one another, in the feature space. This is not really a problem of insufficient minority examples at all; rather, the minority and majority classes are genuinely hard to distinguish using the available features in the region where they overlap. Most classifiers resolve ambiguous regions by favouring whichever class is locally more numerous, which, in an imbalanced dataset, is almost always the majority class. The result is that minority examples sitting in an overlapping region are routinely swallowed by the majority-class prediction, even though, numerically, there might be plenty of minority examples elsewhere in the dataset.

### 5.4 Seeing All Three Together

The figure below generates three small synthetic datasets, one illustrating each characteristic in isolation, so the differences are visible side by side. None of these panels look like a textbook "two well-separated blobs" example; that is deliberate. Real imbalanced datasets routinely look like the middle and right panels, not the left one.

- **Left panel, absolute rarity**: a majority class of 300 points and a minority class of only 8 points. The minority class forms a single tight cluster, but there are so few examples that the cluster's true extent is barely estimable from this sample alone.
- **Middle panel, small disjuncts**: a majority class of 300 points and a minority class of 60 points, but the minority points are split across three separate, small sub-clusters in different corners of the feature space rather than forming one cluster. Each sub-cluster individually behaves like a rarity problem.
- **Right panel, class overlap**: a majority class of 300 points and a minority class of 60 points, both drawn from overlapping regions of the feature space, so that in the central band it is genuinely difficult, even for a human looking at the plot, to tell which class a given point belongs to.


In [ ]:
# Listing 2.
# ── Illustrating absolute rarity, small disjuncts, and class overlap ──────────
# Three small synthetic 2-D datasets, each engineered to isolate one of the
# data-level difficulties described by He and Garcia (2009).

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

maj_colour = 'steelblue'
min_colour = 'tomato'

# ════════════════════════════════════════════════════════════════════════════
# PANEL 1 — Absolute rarity
# A large, well-formed majority class and a tiny minority class: only 8 points.
# ════════════════════════════════════════════════════════════════════════════
X_maj1, _ = make_blobs(n_samples=300, centers=[[0, 0]], cluster_std=1.6, random_state=1)
X_min1, _ = make_blobs(n_samples=8,   centers=[[3.5, 3.5]], cluster_std=0.4, random_state=1)

ax = axes[0]
ax.scatter(X_maj1[:, 0], X_maj1[:, 1], s=18, color=maj_colour, alpha=0.6, label='Majority (n=300)')
ax.scatter(X_min1[:, 0], X_min1[:, 1], s=45, color=min_colour, edgecolors='k', lw=0.5, label='Minority (n=8)')
ax.set_title('Absolute rarity\nminority class too small to estimate', fontsize=10)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.legend(loc='upper left', fontsize=8, framealpha=0.9)

# ════════════════════════════════════════════════════════════════════════════
# PANEL 2 — Small disjuncts
# Minority class is reasonably sized overall (60 points) but fractured into
# three small, disconnected sub-clusters rather than one coherent region.
# ════════════════════════════════════════════════════════════════════════════
X_maj2, _ = make_blobs(n_samples=300, centers=[[0, 0]], cluster_std=1.8, random_state=2)
sub_centres = [[-4.0, 3.0], [4.0, 3.5], [0.5, -4.5]]
X_min2 = np.vstack([
    make_blobs(n_samples=20, centers=[c], cluster_std=0.35, random_state=2)[0]
    for c in sub_centres
])

ax = axes[1]
ax.scatter(X_maj2[:, 0], X_maj2[:, 1], s=18, color=maj_colour, alpha=0.6, label='Majority (n=300)')
ax.scatter(X_min2[:, 0], X_min2[:, 1], s=45, color=min_colour, edgecolors='k', lw=0.5, label='Minority (n=60)')
ax.set_title('Small disjuncts\nminority class split into 3 sub-clusters', fontsize=10)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.legend(loc='upper left', fontsize=8, framealpha=0.9)

# ════════════════════════════════════════════════════════════════════════════
# PANEL 3 — Class overlap
# Majority and minority classes are drawn from overlapping regions, so a
# central band of the feature space contains a genuine mix of both classes.
# ════════════════════════════════════════════════════════════════════════════
X_maj3, _ = make_blobs(n_samples=300, centers=[[0, 0]], cluster_std=2.2, random_state=3)
X_min3, _ = make_blobs(n_samples=60,  centers=[[1.5, 1.5]], cluster_std=1.8, random_state=3)

ax = axes[2]
ax.scatter(X_maj3[:, 0], X_maj3[:, 1], s=18, color=maj_colour, alpha=0.5, label='Majority (n=300)')
ax.scatter(X_min3[:, 0], X_min3[:, 1], s=20, color=min_colour, alpha=0.8, label='Minority (n=60)')
ax.set_title('Class overlap\nminority sits inside the majority region', fontsize=10)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.legend(loc='upper left', fontsize=8, framealpha=0.9)

plt.suptitle(
    'Figure 1: Three intrinsic sources of difficulty in imbalanced data (He & Garcia, 2009)',
    fontsize=12, y=0.98,
)
plt.tight_layout()
plt.show()

Notice that the imbalance ratio alone, roughly 37:1 in the left panel and 5:1 in the other two, does not tell you which of these problems you are facing. A dataset with a mild overall imbalance ratio can still be extremely difficult to learn if its minority examples are fragmented into small disjuncts or buried inside an overlapping region, while a dataset with a severe ratio but a single, well-separated minority cluster can sometimes be handled reasonably well. This is why the He and Garcia (2009) review stresses that **the imbalance ratio by itself is a poor predictor of how hard a dataset will be to learn**: the underlying geometry of the classes matters just as much as their relative counts.


---

## 6. How the Problem Manifests

🔙 [Back to Table of Contents](#Table-of-Contents)

Whatever the underlying cause, whether absolute rarity, small disjuncts, class overlap, or some combination; the practical symptom is almost always the same: **poor recall on the minority class**. Most learning algorithms are, at their core, designed to minimise an aggregate error measure across the whole training set. If 99% of the training examples belong to one class, an algorithm can drive that aggregate error down a very long way simply by getting better and better at the majority class, while making almost no progress on the minority class at all. The overall loss looks small; the minority-class performance is dreadful.

This shows up differently depending on the kind of model:

- **Naïve Bayes** learns a prior $P(Y)$ directly from class frequencies. A 99:1 imbalance means the prior alone already favours the majority class by a factor of 99, before a single feature has been considered. Unless the minority-class likelihoods are extremely distinctive, this prior can dominate the posterior calculation for almost every point.
- **Decision trees** select splits that maximise information gain or minimise Gini impurity averaged across all the training examples that reach a node. With a heavily skewed class distribution, a tree can often achieve a very low impurity score simply by predicting the majority class everywhere; there is little incentive for the greedy search to carve out small regions for the minority class.
- **Logistic regression and linear SVMs** minimise an aggregate loss (log-loss or hinge loss) summed across all training points. Because there are vastly more majority points, the gradient of that aggregate loss is dominated by how well the model is doing on the majority class, and minority points contribute comparatively little pull on the final decision boundary.

The common thread is that **none of these algorithms know, by default, that the minority class matters more than its numbers suggest**. They are simply doing what they were built to do: minimise an aggregate error. The result is a decision boundary that sits very close to the majority class, conceding the entire region around and including most minority points in exchange for a tiny reduction in aggregate error.

This is precisely why precision, recall, and F1-score broken down **per class** are essential diagnostic tools here. A model with 99% overall accuracy but a recall of 0.04 on the minority class is not a good model; it has simply learned to ignore the class that matters. Comparing precision and recall specifically on the minority class, rather than looking at accuracy alone, is the standard way to detect this failure mode early, before a model with this kind of blind spot is deployed.


---

## 7. Imbalanced Learning, Anomaly Detection, and One-Class Learning

🔙 [Back to Table of Contents](#Table-of-Contents)

Imbalanced learning sits on a spectrum, rather than being a single, sharply defined problem. As the imbalance ratio grows more and more extreme, and the minority class shrinks further and further, the problem gradually shades into territory usually described using different terminology altogether: **anomaly detection**, **outlier detection**, and **one-class learning**.

The underlying connection is straightforward. At a 9:1 ratio, we usually still think of this as a two-class classification problem with a minority class, and the methods covered in this notebook (resampling, cost-sensitive learning) are the natural tools to reach for. At a 10,000:1 ratio, with perhaps only a handful of confirmed positive examples ever observed, framing the problem as "classify into two classes" starts to break down. There may be too few positive examples to characterise what the minority class even looks like in feature space, which is exactly the absolute rarity problem from Section 5.1, taken to its logical extreme.

In this extreme regime, a common reframing is to stop trying to model the minority class directly and instead model **only the majority ("normal") class**. The model learns what normal data looks like, and anything that does not fit that learned profile, anything that scores as sufficiently unusual, is flagged as a potential anomaly or outlier. This is the **one-class learning** paradigm: rather than learning a boundary between two classes, the model learns a boundary around a single class and treats everything outside it as anomalous. One-Class SVM and Isolation Forest are two well-known examples of algorithms built around this idea; we do not cover them in detail in this notebook, but it is worth recognising the conceptual link.

The practical takeaway is this: when you encounter a severely imbalanced dataset, it is worth asking which framing fits your problem better.

- If you have a reasonable number of minority examples, even if heavily outnumbered, and you want a model that explicitly distinguishes between two known classes, the **two-class imbalanced learning** approach covered in the rest of this notebook (resampling, cost-sensitive learning) is usually the right starting point.
- If positive examples are extremely scarce, poorly characterised, or the very notion of "what a positive example looks like" is fuzzy or evolving (fraud patterns change over time, for example), an **anomaly detection** or **one-class learning** framing, where you model normality and flag deviations from it, is often more robust.

These are not mutually exclusive; in practice, many production fraud and intrusion detection systems combine both ideas, using anomaly scores as an additional feature fed into a supervised, imbalance-aware classifier.


---

## 8. Data-Level Approaches

🔙 [Back to Table of Contents](#Table-of-Contents)

The most direct way to address class imbalance is to change the **data**, not the algorithm: rebalance the training set before fitting any model, so that the learning algorithm sees a more even mix of classes and is no longer implicitly rewarded for ignoring the minority class. This family of techniques is collectively known as **resampling**, and it is usually the first thing practitioners try because it requires no changes to the downstream model at all; any classifier can be trained on the resampled data exactly as before.

There are three broad resampling strategies: removing majority examples (**undersampling**), duplicating or synthesising minority examples (**oversampling**), and ensuring that any train/test split preserves the original class proportions in both subsets (**stratification**, which is not strictly a resampling technique but is essential groundwork for evaluating one fairly).

### 8.1 Random Undersampling

**Random undersampling** rebalances a dataset by randomly discarding examples from the majority class until the two classes are closer to parity. If a dataset has 9,900 majority examples and 100 minority examples, random undersampling might randomly keep only 100 to 300 of the majority examples, throwing the rest away, so that the resulting training set is balanced or only mildly imbalanced.

**Pros:**

- Simple to implement and very fast, since it only involves removing rows.
- Reduces the size of the training set, which can speed up training considerably on very large datasets.
- Does not introduce any artificial data points; every remaining example is a genuine observation.

**Cons:**

- Discards potentially useful information: majority-class examples near the decision boundary, which are often the most informative for learning where that boundary should sit, might be removed purely by chance.
- With a severe imbalance ratio, undersampling down to parity can leave a very small overall training set, reintroducing an absolute rarity problem for the majority class.
- Different random seeds produce different subsets of the majority class, so results can vary between runs unless this is controlled for.

Performing random undersampling with `imbalanced-learn` follows the same `fit_resample` pattern used by every sampler in this library:

```python
from imblearn.under_sampling import RandomUnderSampler

# sampling_strategy='auto' undersamples the majority class down to
# match the size of the minority class (a 1:1 ratio)
rus = RandomUnderSampler(sampling_strategy='auto', random_state=0)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)
```

### 8.2 Random Oversampling

**Random oversampling** takes the opposite approach: rather than removing majority examples, it randomly duplicates minority examples until the two classes are closer to parity. With 9,900 majority and 100 minority examples, random oversampling might draw repeatedly, with replacement, from the 100 minority examples until there are roughly 9,900 minority rows in the resampled training set, many of them exact copies of one another.

**Pros:**

- No information is discarded: every majority example remains in the training set.
- Simple and fast to apply.
- Particularly useful when the dataset is already small overall, since undersampling would shrink it further.

**Cons:**

- Because minority examples are exact duplicates rather than new information, the model can overfit to those specific repeated points rather than learning the genuine shape of the minority class.
- The training set grows substantially in size, which increases training time, sometimes considerably so for slower algorithms such as SVMs.
- Does nothing to address class overlap: duplicated minority points sitting inside an overlapping region are still exactly as ambiguous as before, just more numerous.

```python
from imblearn.over_sampling import RandomOverSampler

# sampling_strategy='auto' oversamples the minority class up to
# match the size of the majority class (a 1:1 ratio)
ros = RandomOverSampler(sampling_strategy='auto', random_state=0)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)
```

### 8.3 Stratification

Resampling addresses the **training set**, but there is a related and equally important issue when **splitting** data into training and test sets in the first place. If a train/test split is performed by drawing rows uniformly at random, it is entirely possible, especially with a small minority class, to end up with a test set that contains very few minority examples, or even none at all. A handful of minority examples in the test set gives an extremely noisy estimate of minority-class recall: with only 5 true positive cases in a held-out set, getting one extra prediction right or wrong swings recall by 20 percentage points.

**Stratified splitting** solves this by ensuring that the class proportions in the training set and the test set both match the proportions in the full dataset. This does not change the imbalance ratio itself, but it guarantees that both splits are representative, so that performance estimates on the test set are meaningful. This is exactly what the `stratify=y` argument to `train_test_split` does, and it is something we have used throughout earlier notebooks in this series even outside the context of imbalance:

```python
from sklearn.model_selection import train_test_split

# stratify=y ensures the train and test sets both preserve
# the original class proportions of the full dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)
```

A useful rule of thumb: **resampling should only ever be applied to the training set, never to the test set.** The test set must remain an honest, untouched sample of the real-world class distribution, so that the reported performance reflects what the model will actually encounter in deployment. Resampling the test set as well would give an artificially optimistic, and misleading, picture of how well the model generalises.

### 8.4 Artificial Rare-Class Generation: SMOTE and Borderline-SMOTE

Random oversampling has an obvious weakness: duplicating existing minority points adds no new information, it only adds repetition. **SMOTE** (Synthetic Minority Over-sampling Technique), introduced by Chawla et al. (2002), addresses this by generating genuinely new, synthetic minority examples rather than copying existing ones.

The idea is geometric and fairly simple. For each minority example $\mathbf{x}_i$ that needs to be oversampled, SMOTE:

1. Finds its $k$ nearest minority-class neighbours in feature space (a common default is $k=5$).
2. Randomly picks one of those neighbours, $\mathbf{x}_{zi}$.
3. Generates a new synthetic point somewhere along the straight line connecting $\mathbf{x}_i$ and $\mathbf{x}_{zi}$:

$$\mathbf{x}_{\text{new}} = \mathbf{x}_i + \lambda \cdot (\mathbf{x}_{zi} - \mathbf{x}_i), \qquad \lambda \sim \text{Uniform}(0, 1)$$

In plain English: SMOTE does not just copy an existing minority point, it invents a brand new point that sits somewhere between two real minority examples that are already close together. Repeating this process many times fills in the space around the minority class with plausible new examples, rather than piling up duplicates on top of the originals.

**Pros:**

- Generates genuinely new data points rather than duplicates, which tends to reduce the overfitting seen with random oversampling.
- Helps the classifier learn a more general region for the minority class, rather than memorising a small number of repeated points.
- Works well when the minority class forms a reasonably coherent region, where interpolating between nearby points produces sensible new examples.

**Cons:**

- SMOTE interpolates blindly between nearest neighbours without checking whether the new point falls inside a region that overlaps with the majority class. In an overlapping region (Section 5.3), this can generate synthetic minority points that sit deep inside majority territory, making the overlap problem worse rather than better.
- It assumes minority neighbours are meaningfully close in feature space; with small disjuncts (Section 5.2), SMOTE can interpolate between points from two different sub-clusters, generating synthetic examples in the empty space between them that do not correspond to any real region of the minority class.
- Like all oversampling methods, it increases the size of the training set and therefore training time.

**Borderline-SMOTE**, a widely used variant, was designed specifically to address the overlap weakness above. Rather than treating every minority point equally, it first identifies which minority points are "in danger": points whose nearest neighbours are predominantly from the majority class, meaning they sit close to, or just inside, the decision boundary. Borderline-SMOTE then concentrates its synthetic sample generation around these borderline points rather than the minority class as a whole, on the reasoning that it is precisely the borderline region where extra synthetic examples are most useful for sharpening the decision boundary, while interior minority points, already easy to classify, gain comparatively little from additional synthetic neighbours.

Both SMOTE and Borderline-SMOTE are available in `imbalanced-learn` with an almost identical interface to the random samplers above:

```python
from imblearn.over_sampling import SMOTE, BorderlineSMOTE

# SMOTE: generate synthetic minority points by interpolating
# between each minority example and one of its k nearest minority neighbours
smote = SMOTE(sampling_strategy='auto', k_neighbors=5, random_state=0)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# Borderline-SMOTE: as above, but concentrates synthetic points
# around minority examples that sit near the majority-class boundary
bsmote = BorderlineSMOTE(sampling_strategy='auto', k_neighbors=5, random_state=0)
X_resampled_b, y_resampled_b = bsmote.fit_resample(X_train, y_train)
```

It is worth noting that this notebook is not attempting a complete review of resampling techniques. `imbalanced-learn` also implements several other oversampling variants (`ADASYN`, `SVMSMOTE`, `KMeansSMOTE`), more sophisticated undersampling methods that remove majority points selectively rather than at random (`NearMiss`, `EditedNearestNeighbours`, `TomekLinks`), and combined methods that oversample and then clean up the result (`SMOTEENN`, `SMOTETomek`). The two methods covered here, plain SMOTE and Borderline-SMOTE, are simply the most widely used starting points.

The code cell below applies plain SMOTE to a small synthetic dataset with a 50:1 imbalance ratio and visualises the result: the original minority points alongside the new synthetic points SMOTE has generated around them.


In [ ]:
# Listing 3.
# ── SMOTE on a synthetic two-class dataset ────────────────────────────────────
# We create a deliberately imbalanced 2-D dataset (50:1 ratio), apply SMOTE,
# and plot the original points alongside the newly generated synthetic ones.

X_demo, y_demo = make_classification(
    n_samples=1020,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.98, 0.02],   # 98% majority, 2% minority — a 50:1 imbalance ratio
    class_sep=1.2,
    random_state=7,
)

print('Class counts before SMOTE:', dict(pd.Series(y_demo).value_counts()))

# Apply SMOTE to fully balance the two classes
smote_demo = SMOTE(sampling_strategy='auto', k_neighbors=5, random_state=7)
X_smote, y_smote = smote_demo.fit_resample(X_demo, y_demo)

print('Class counts after SMOTE: ', dict(pd.Series(y_smote).value_counts()))

# Identify which minority points in the resampled set are newly synthesised
# (i.e. were not present in the original data) purely for visualisation purposes
n_original_minority = (y_demo == 1).sum()
n_total_minority_after = (y_smote == 1).sum()
n_synthetic = n_total_minority_after - n_original_minority

fig, ax = plt.subplots(figsize=(7, 6))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

# Majority class (unaffected by SMOTE, which only oversamples the minority class)
ax.scatter(X_demo[y_demo == 0, 0], X_demo[y_demo == 0, 1],
           s=14, color='steelblue', alpha=0.4, label=f'Majority (n={(y_demo==0).sum()})')

# Original minority points (the first n_original_minority minority rows are unchanged)
orig_min_mask = np.zeros(len(y_smote), dtype=bool)
orig_min_mask[np.where(y_smote == 1)[0][:n_original_minority]] = True
ax.scatter(X_smote[orig_min_mask, 0], X_smote[orig_min_mask, 1],
           s=60, color='tomato', edgecolors='k', lw=0.7, zorder=3,
           label=f'Original minority (n={n_original_minority})')

# Synthetic minority points generated by SMOTE
synth_mask = (y_smote == 1) & (~orig_min_mask)
ax.scatter(X_smote[synth_mask, 0], X_smote[synth_mask, 1],
           s=30, color='gold', edgecolors='k', lw=0.3, alpha=0.85, zorder=2,
           label=f'SMOTE synthetic (n={n_synthetic})')

ax.set_title('Figure 2: SMOTE oversampling on a 50:1 imbalanced dataset', fontsize=12)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

Notice how the gold synthetic points cluster tightly around the original red minority points, filling in the space between nearby minority neighbours rather than scattering randomly across the plot. This is the interpolation step at work: every synthetic point lies somewhere on a line segment connecting two real minority examples, which is exactly what the SMOTE formula in Section 8.4 describes. Compare this with what random oversampling would have produced: the same handful of original red points, simply stacked on top of each other many times over, with no new spatial coverage at all.


---

## 9. Algorithm-Level Approaches

🔙 [Back to Table of Contents](#Table-of-Contents)

Data-level approaches change the **training data** so that a standard, unmodified learning algorithm behaves better on it. **Algorithm-level approaches** take the opposite route: they leave the training data untouched and instead change **what the learning algorithm is optimising for**, so that it is no longer implicitly indifferent to which class its mistakes fall on.

### 9.1 Changing the Learning Objective

Recall from Section 6 that most classifiers minimise an aggregate error measure summed or averaged across every training example, treating every misclassification as equally costly. Algorithm-level approaches challenge that assumption directly: they modify the objective function itself so that mistakes on the minority class are penalised more heavily than mistakes on the majority class. Rather than asking the algorithm to minimise *"how many mistakes did I make?"*, we ask it to minimise *"how much did my mistakes cost?"*, where the cost of a missed minority example is set deliberately higher than the cost of a missed majority example.

This single idea, weighting errors unequally according to which class they affect, can be implemented in many different ways depending on the model family: by modifying the loss function directly (as in cost-sensitive logistic regression or cost-sensitive SVMs), by reweighting how impurity is calculated at each split in a decision tree, by adjusting the prior probabilities used by a generative model such as Naïve Bayes, or by reweighting how individual trees vote in an ensemble. Surveying every one of these variants for every algorithm family is well beyond the scope of this notebook; entire review papers are devoted to the topic. Instead, the most general and widely applicable version of this idea, **cost-sensitive learning**, is introduced below, followed by a single concrete worked example showing exactly how it changes the behaviour of a model we have already covered in detail: Naïve Bayes.

### 9.2 Cost-Sensitive Learning

**Cost-sensitive learning** formalises the idea above using a **cost matrix**: a table specifying the cost of every possible combination of true label and predicted label. For binary classification this is a 2×2 matrix:

| | Predicted Negative | Predicted Positive |
|---|---|---|
| **Actually Negative** | $C_{00}$ (true negative) | $C_{01}$ (false positive) |
| **Actually Positive** | $C_{10}$ (false negative) | $C_{11}$ (true positive) |

In standard, cost-insensitive learning, every type of error is implicitly assigned equal cost: $C_{01} = C_{10}$, and correct predictions cost nothing. Cost-sensitive learning instead sets $C_{10}$, the cost of a **false negative**, missing a true minority-class example, much higher than $C_{01}$, the cost of a **false positive**, raising a false alarm on a majority-class example. This directly mirrors the real-world asymmetry in most imbalanced problems: failing to flag a fraudulent transaction or a malignant tumour is typically far more costly than incorrectly flagging a legitimate one.

The practical effect of a high $C_{10}$ is that the learning algorithm becomes willing to accept more false positives in exchange for catching more true positives, shifting the decision boundary further into what was previously majority-class territory. This is a direct, controllable trade along the precision-recall curve: increasing the cost of missing the minority class increases minority-class recall, generally at some cost to majority-class precision.

A common and convenient simplification used throughout scikit-learn is to set **class weights** rather than a full cost matrix: each class is assigned a single weight, and the algorithm's objective function is adjusted so that errors on a higher-weighted class count for more. Setting `class_weight='balanced'` on most scikit-learn classifiers automatically sets each class's weight inversely proportional to its frequency in the training data, so that the rare class is weighted up and the common class is weighted down, without needing to specify exact costs by hand:

```python
from sklearn.linear_model import LogisticRegression

# class_weight='balanced' automatically weights each class inversely
# proportional to its frequency, so the minority class counts for more
# in the loss function without any resampling of the training data
clf = LogisticRegression(class_weight='balanced')
clf.fit(X_train, y_train)
```

**Pros of cost-sensitive learning:**

- Requires no changes to the training data: the original dataset, with its original size and class proportions, is used directly.
- Avoids the duplication or synthetic-point issues associated with oversampling, and avoids discarding data the way undersampling does.
- The cost ratio gives an explicit, interpretable knob for controlling the precision-recall trade-off, which can often be set directly from real business or clinical costs rather than guessed at.

**Cons of cost-sensitive learning:**

- Not every algorithm or every library implementation exposes a convenient way to set class weights or a custom cost matrix; support varies considerably across model families and toolkits.
- The correct cost ratio is not always obvious. In well-defined problems such as credit scoring it can sometimes be estimated from real financial costs, but in many other settings it has to be tuned experimentally, similar to any other hyperparameter.
- It addresses the symptom (the algorithm's indifference to which class it gets wrong) without addressing underlying data difficulties such as small disjuncts or class overlap; a poorly characterised minority class is still poorly characterised, however heavily its errors are weighted.

### 9.3 A Worked Example: Cost-Sensitive Naïve Bayes

Recall from Notebook 6 that Naïve Bayes classifies a new point by computing, for each class $y$, a score proportional to the posterior probability, and picking whichever class has the largest score:

$$\hat{y} = \underset{y}{\arg\max} \ P(Y = y) \cdot \prod_{i=1}^{n} P(x_i \mid Y = y)$$

The prior $P(Y = y)$ is, by default, estimated directly from the class frequencies in the training data. With a 50:1 imbalance ratio, $P(Y = \text{majority}) \approx 0.98$ and $P(Y = \text{minority}) \approx 0.02$, so the minority class starts every single prediction at an enormous disadvantage before a single feature value has even been considered.

A simple and transparent way to make Naïve Bayes cost-sensitive is to **replace the data-driven prior with an artificially adjusted one**, directly reflecting the relative cost of the two error types rather than the relative frequency of the two classes. Instead of using $P(Y=\text{minority}) \approx 0.02$, we substitute a much larger artificial prior, effectively telling the model: *"treat the minority class as if missing it were costly enough to outweigh its rarity in the training data."* The feature likelihoods $P(x_i \mid Y=y)$, learned from the data as before, are left completely untouched; only the prior, the term that encodes how the model weighs the two classes before looking at any evidence, is adjusted.

This is mathematically equivalent to scaling the posterior scores of each class by a chosen cost factor, which is exactly what `class_weight`-based cost-sensitive learning does for other scikit-learn estimators, just made fully explicit here so the mechanism is easy to see.

The code cell below trains a standard Naïve Bayes classifier and a cost-sensitive variant, with an artificially boosted minority-class prior, on the same imbalanced dataset, and compares their recall and precision on the minority class directly.


In [ ]:
# Listing 4.
# ── Cost-sensitive Naïve Bayes via an adjusted prior ───────────────────────────
# We compare a standard GaussianNB classifier against a cost-sensitive variant
# that replaces the data-driven minority-class prior with an artificially
# raised one, simulating a high cost for missing a true minority example.

X_cs, y_cs = make_classification(
    n_samples=2000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.95, 0.05],   # 95:5, a 19:1 imbalance ratio
    class_sep=1.0,
    random_state=11,
)

X_tr_cs, X_te_cs, y_tr_cs, y_te_cs = train_test_split(
    X_cs, y_cs, test_size=0.25, random_state=11, stratify=y_cs
)

print(f'Training set class counts : {dict(pd.Series(y_tr_cs).value_counts())}')
print(f'Test set class counts     : {dict(pd.Series(y_te_cs).value_counts())}')
print()

# ════════════════════════════════════════════════════════════════════════════
# MODEL 1 — Standard Naïve Bayes
# The prior P(Y) is learned directly from the training class frequencies.
# ════════════════════════════════════════════════════════════════════════════
nb_standard = GaussianNB()
nb_standard.fit(X_tr_cs, y_tr_cs)
y_pred_standard = nb_standard.predict(X_te_cs)

print('Standard Naïve Bayes')
print('=' * 50)
print(f'Learned priors P(Y) : {dict(zip(nb_standard.classes_, np.round(nb_standard.class_prior_, 4)))}')
print(f'Minority recall     : {recall_score(y_te_cs, y_pred_standard, pos_label=1):.3f}')
print(f'Minority precision  : {precision_score(y_te_cs, y_pred_standard, pos_label=1):.3f}')
print(f'Minority F1-score   : {f1_score(y_te_cs, y_pred_standard, pos_label=1):.3f}')
print()

# ════════════════════════════════════════════════════════════════════════════
# MODEL 2 — Cost-sensitive Naïve Bayes
# We fit GaussianNB as normal (to learn the feature likelihoods), then
# manually overwrite class_prior_ with an artificially raised minority prior,
# simulating a high cost for a false negative on the minority class.
# This does NOT change the learned likelihoods — only the prior term.
# ════════════════════════════════════════════════════════════════════════════
nb_cost_sensitive = GaussianNB()
nb_cost_sensitive.fit(X_tr_cs, y_tr_cs)

# Replace the data-driven prior with an artificial cost-weighted prior.
# Here we treat the minority class as 6x more important than its raw
# frequency would suggest, then renormalise so the priors sum to 1.
cost_weight_minority = 6.0
raw_priors = nb_cost_sensitive.class_prior_.copy()
adjusted_priors = raw_priors.copy()
adjusted_priors[1] *= cost_weight_minority   # boost the minority class prior
adjusted_priors /= adjusted_priors.sum()     # renormalise to sum to 1

nb_cost_sensitive.class_prior_ = adjusted_priors
y_pred_cost_sensitive = nb_cost_sensitive.predict(X_te_cs)

print('Cost-Sensitive Naïve Bayes (minority prior boosted x6)')
print('=' * 50)
print(f'Original priors P(Y)  : {dict(zip(nb_cost_sensitive.classes_, np.round(raw_priors, 4)))}')
print(f'Adjusted priors P(Y)  : {dict(zip(nb_cost_sensitive.classes_, np.round(adjusted_priors, 4)))}')
print(f'Minority recall       : {recall_score(y_te_cs, y_pred_cost_sensitive, pos_label=1):.3f}')
print(f'Minority precision    : {precision_score(y_te_cs, y_pred_cost_sensitive, pos_label=1):.3f}')
print(f'Minority F1-score     : {f1_score(y_te_cs, y_pred_cost_sensitive, pos_label=1):.3f}')

Look closely at the comparison: the cost-sensitive model's minority recall should be noticeably higher than the standard model's, at some cost to minority precision. This is exactly the trade-off described in Section 9.2. By artificially inflating $P(Y=\text{minority})$, we have told the model to require less convincing evidence from the features before it predicts the minority class, since the prior is doing more of the work. The model now catches more genuine minority cases (higher recall), but it also raises more false alarms on majority-class points that happen to sit close to the boundary (lower precision).

Crucially, notice what we did **not** change: the feature likelihoods learned during `fit()`, the means and variances describing what each class's features actually look like, are completely identical between the two models. The two classifiers have learned exactly the same picture of the data; they differ only in how heavily that evidence is weighed against the prior before reaching a final decision. This is the essence of algorithm-level, cost-sensitive learning: the data and the model's internal representation of it are untouched, but the *decision rule* built on top of that representation has been deliberately rebalanced.

The cost weight of 6 used above was chosen arbitrarily for illustration. In practice this value should be informed by the real-world cost ratio of the two error types wherever that is available, for example, the expected financial loss from a missed fraud case divided by the cost of investigating a false alarm, or set through experimentation by sweeping a range of values and tracking minority-class recall and precision, much as we tuned `C` for the SVM in Notebook 6.



---

## 10. Summary

🔙 [Back to Table of Contents](#Table-of-Contents)

| Concept | What it describes |
|---------|-------------------|
| **Class imbalance** | A dataset where one class (the majority) substantially outnumbers another (the minority), often summarised by the imbalance ratio. |
| **Absolute rarity** | The minority class has too few examples overall to reliably characterise its shape, regardless of the imbalance ratio. |
| **Small disjuncts** | The minority class is fragmented into several small, disconnected sub-clusters rather than one coherent region. |
| **Class overlap** | Minority and majority examples occupy the same region of feature space, making them genuinely hard to distinguish. |
| **Poor minority recall** | The typical symptom of imbalance: standard classifiers minimise aggregate error and so favour the majority class, often missing most minority cases while still posting high overall accuracy. |
| **Anomaly / outlier detection, one-class learning** | The natural extension of imbalanced learning as the minority class becomes vanishingly rare: instead of learning a two-class boundary, model the majority class and flag deviations from it. |
| **Random undersampling** | Discard majority examples at random to rebalance the training set. Fast and simple, but can discard useful information. |
| **Random oversampling** | Duplicate minority examples at random to rebalance the training set. Loses no information, but can cause overfitting to repeated points. |
| **Stratification** | Ensure train/test splits preserve the original class proportions, so evaluation on the minority class is reliable. Should always be combined with the techniques above, applied only to the training set. |
| **SMOTE** | Generates new synthetic minority examples by interpolating between a minority point and one of its nearest minority neighbours. |
| **Borderline-SMOTE** | A SMOTE variant that concentrates synthetic sample generation around minority points near the majority-class boundary, where extra examples are most useful. |
| **Cost-sensitive learning** | An algorithm-level approach: change the learning objective so that errors on the minority class are penalised more heavily, e.g. via `class_weight='balanced'` or an adjusted prior, rather than resampling the data. |

Data-level and algorithm-level approaches are not mutually exclusive; in practice it is common to combine resampling (such as SMOTE) with a cost-sensitive model, and to compare several combinations using minority-class precision, recall, and F1-score, since accuracy alone, as Section 4 showed, can be dangerously misleading on imbalanced data.



---


## 11. References

🔙 [Back to Table of Contents](#Table-of-Contents)

Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic minority over-sampling technique. *Journal of Artificial Intelligence Research*, 16, 321–357. https://doi.org/10.1613/jair.953

Han, H., Wang, W. Y., & Mao, B. H. (2005). Borderline-SMOTE: A new over-sampling method in imbalanced data sets learning. In *International Conference on Intelligent Computing* (pp. 878–887). Springer. https://doi.org/10.1007/11538059_91

He, H., & Garcia, E. A. (2009). Learning from imbalanced data. *IEEE Transactions on Knowledge and Data Engineering*, 21(9), 1263–1284. https://doi.org/10.1109/TKDE.2008.239

Holte, R. C., Acker, L. E., & Porter, B. W. (1989). Concept learning and the problem of small disjuncts. In *Proceedings of the 11th International Joint Conference on Artificial Intelligence* (pp. 813–818). Morgan Kaufmann.

Japkowicz, N. (2004). Class imbalances versus small disjuncts. *ACM SIGKDD Explorations Newsletter*, 6(1), 40–49. https://doi.org/10.1145/1007730.1007737

Lemaître, G., Nogueira, F., & Aridas, C. K. (2017). Imbalanced-learn: A Python toolbox to tackle the curse of imbalanced datasets in machine learning. *Journal of Machine Learning Research*, 18(17), 1–5. http://jmlr.org/papers/v18/16-365.html

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830. http://jmlr.org/papers/v12/pedregosa11a.html